

**CIFAR10: 60,000 32x32 RGB-color images: airplane (0), automobile (1), bird (2), cat (3), deer (4), dog (5), frog (6), horse (7), ship (8), and truck (9).

Data is split into 5 training batches, one test batch


In [13]:
# Imports 
import numpy as np, pandas as pd
# For image processing
from PIL import Image
from IPython.display import Image

import os 
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt


In [14]:
"""
# Where we are right now, absolute pathing
base_dir = os.path.dirname(__file__)
images_file_path = os.path.join(base_dir, "CCSN_Image_set")
# example path for cirrus clouds
ci_file_path = os.path.join(images_file_path, "Ci")
"""

'\n# Where we are right now, absolute pathing\nbase_dir = os.path.dirname(__file__)\nimages_file_path = os.path.join(base_dir, "CCSN_Image_set")\n# example path for cirrus clouds\nci_file_path = os.path.join(images_file_path, "Ci")\n'

In [15]:
# Tensor ~ vector (array of components)
# Compose allows for list of several transformations simultanously 
transform = transforms.Compose([
    # Input images as 0-255 RGB-values, we want in range [-1,1]
    # Converts PIL [0,255] to [0.0, 1.0]
    transforms.ToTensor(),
    # Adjusts with mean=0.5, stdv=0.5 
    # 3 Channels because of RGB (Greyscale would be 1 channel)
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

In [16]:
train_data = torchvision.datasets.CIFAR10(root="./data", train=True, transform=transform, download=True)
test_data = torchvision.datasets.CIFAR10(root="./data", train=False, transform=transform, download=True)

# 
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=True, num_workers=2)

In [17]:
image, label = train_data[0]
# 3 channel 32x32 images
image.size()

torch.Size([3, 32, 32])

In [18]:
# Labels
class_names = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

In [ ]:
Image("2D-CNN-layer-calculation.png")

class KulingNet(nn.Module):
    # Create base nn-architecture
    def __init__(self):
        super().__init__() 
        # 5x5 kernel moves across images, creates feature maps
        # s=1, k=5, nin=32, p=0
        self.conv1 = nn.Conv2d(3, 12, 5) # (12, 28, 28)
        # Pools together 2x2 pixels to single pixel (extracts important features)
        self.pool = nn.MaxPool2d(2,2) # (12,14,14)
        # s=1, k=5, nin=14, p=0
        self.conv2 = nn.Conv2d(12, 24, 5) # (24,10,10) -> (24,5,5) -> Flatten (24,5,5)
        # "Dense" layers, neurons have to be compatible, but chosen freely
        self.fc1 = nn.Linear(24*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84,10)
    
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x
    
    

In [ ]:
net = KulingNet()
loss_function = nn.CrossEntropyLoss()
# lr: step size during gradient descent
# momentum: using past gradients (inertia)
optimizer = optim.SGD(net.parameters(), lr = 0.001, momentum=0.9)


In [ ]:
"""
for epoch in range(30):
    print("Epoch:", epoch, "...")
    running_loss = 0.0
    for i, data in enumerate(train_loader):
        # each batch contains data
        inputs, labels = data

        optimizer.zero_grad()

        outputs = net(inputs)
        # Calculate difference of predicted vs actual labels
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    print(f"loss: {running_loss / len(train_loader):.4f}")

"""

Epoch: 0 ...
loss: 2.2037
Epoch: 1 ...
loss: 1.7616
Epoch: 2 ...
loss: 1.5331
Epoch: 3 ...
loss: 1.4012
Epoch: 4 ...
loss: 1.3050
Epoch: 5 ...
loss: 1.2176
Epoch: 6 ...
loss: 1.1354
Epoch: 7 ...
loss: 1.0721
Epoch: 8 ...
loss: 1.0167
Epoch: 9 ...
loss: 0.9702
Epoch: 10 ...
loss: 0.9277
Epoch: 11 ...
loss: 0.8911
Epoch: 12 ...
loss: 0.8547
Epoch: 13 ...
loss: 0.8242
Epoch: 14 ...
loss: 0.7902
Epoch: 15 ...
loss: 0.7612
Epoch: 16 ...
loss: 0.7314
Epoch: 17 ...
loss: 0.7039
Epoch: 18 ...
loss: 0.6781
Epoch: 19 ...
loss: 0.6536
Epoch: 20 ...
loss: 0.6311
Epoch: 21 ...
loss: 0.6071
Epoch: 22 ...
loss: 0.5865
Epoch: 23 ...
loss: 0.5643
Epoch: 24 ...
loss: 0.5386
Epoch: 25 ...
loss: 0.5214
Epoch: 26 ...
loss: 0.5014
Epoch: 27 ...
loss: 0.4843
Epoch: 28 ...
loss: 0.4636
Epoch: 29 ...
loss: 0.4402


In [22]:
# Save net parameters
torch.save(net.state_dict(), "trained_net.pth")


In [23]:
# Load net with parameters
net = KulingNet()
net.load_state_dict(torch.load("trained_net.pth"))

<All keys matched successfully>

In [24]:
# Checking the network
correct = 0
total = 0 

net.eval()
# no gradient descent during evaluation, just feed input forward
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _ , predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy} %")


Accuracy: 69.96 %


In [26]:
# Transform test images with different format 
new_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

def load_image(path):
    image = Image.open(path)
    image = new_transform(image)
    # Add batch dimension
    image = image.unsqueeze(0)
    return image

paths = []
images = [load_image(img) for img in paths]

net.eval()
with torch.no_grad():
    for image in images:
        output = net(image)
        _, predicted = torch.max(output,1)
        print(f"Prediction: {class_names[predicted.item()]}")

